In [7]:
# Micrograd (Back Prop and Sampling)

In [4]:
from math import exp, log

In [5]:
class Value:
    def __init__ (self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = _children
        self._op = _op

    def __repr__ (self):
        return f'Value(data={self.data})'

    def __add__ (self, other):  
        other = other if isinstance(other, Value) else Value(other)   # Other is a Value Instance else create it
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out

    def __radd__(self, other):
        return self + other

    def __sub__ (self, other):
        other = other if isinstance(other, Value) else Value(other)   # Other is a Value Instance else create it
        out = Value(self.data - other.data, (self, other), '-')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward

        return out

    def __rsub__ (self, other):
        return self - other

    def __mul__ (self, other):
        other = other if isinstance(other, Value) else Value(other)   # Other is a Value Instance else create it
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __rmul__(self, other):
        return self * other

    def __pow__ (self, other):
        assert isinstance(other, (int, float)), "only supports int and float powers"
        out = Value(self.data ** other, (self, ), f'**{other}')

        def _backward():
            self.grad += (other * self.data**(other - 1)) * out.grad
        out._backward = _backward

        return out

    def __truediv__ (self, other):
        return self * other**-1

    def tanh(self):
        x = self.data
        t = (exp(2*x) - 1)/(exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward

        return out
        
    def backward(self):
        topo = []
        visited = set()
        def build_topo(node):      # arrangement in topological order for back prop
            if node not in visited:
                visited.add(node)
                for child in node._prev:
                    build_topo(child)
                topo.append(node)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

In [6]:
# sigle neuron back prop

# ---data---
x1 = Value(2.0)
x2 = Value(0.0)

# ---weights---
w1 = Value(-3.0)
w2 = Value(1.0) 

x1w1 = x1 * w1
x2w2 = x2 * w2

b = Value(6.8813735870195432) # ---bais---

x1w1x2w2 = x1w1 + x2w2

n = x1w1x2w2 + b
o = n.tanh()  # tanh linearity

o.backward()

print(x1.grad, w1.grad, x2.grad, w2.grad)
print(x1w1.grad, x2w2.grad)
print(x1w1x2w2.grad)
print(n.grad)
print(o.grad)

-1.4999999999999996 0.9999999999999998 0.4999999999999999 0.0
0.4999999999999999 0.4999999999999999
0.4999999999999999
0.4999999999999999
1.0


In [7]:
!pip install torch

In [16]:
# Implimenting same using PyTorch

import torch 

x1 = torch.Tensor([2.0]).double()                          ; x1.requires_grad = True
x2 = torch.Tensor([0.0]).double()                          ; x2.requires_grad = True
w1 = torch.Tensor([-3.0]).double()                         ; w1.requires_grad = True
w2 = torch.Tensor([1.0]).double()                          ; w2.requires_grad = True
b = torch.Tensor([6.8813735870195432]).double()            ; b.requires_grad = True

n = x1*w1 + x2*w2 + b
o = torch.tanh(n)

print(o.data.item())
o.backward()

print(x1.grad.item())
print(w1.grad.item())
print(x2.grad.item())
print(w2.grad.item())

0.7071066904050358
-1.5000003851533106
1.0000002567688737
0.5000001283844369
0.0


In [21]:
import random

class Neuron:  # defines a singhle neuron
    
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]  # random double b/w -1 to 1
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):   # let n = Neuron(...) then n(x) will return the output of this function
        act = sum((w*x for w, x in zip(self.w, x)), self.b)
        out = act.tanh()     # Normalized
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:   # to create a number of neurons

    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:    # to create a multi layer perceptron

    def __init__(self, nin, nouts):   # list of nout
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x 

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [ ]:
def msl( yactual, ypred ):         # mean squared loss
    return sum((ya - yp)**2 for ya, yp in zip(yactual, ypred))

def update(parameters, h):         # h is the learning rate
    for p in parameters:
        p.data += -h * p.grad

In [23]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 1.0, 1.0],
    [1.0, 1.0, -1.0],
]
ys = [1.0, -1.0, -1.0, 1.0] # desired targets

mlp = MLP(3, [4, 4, 1])

In [65]:
for k in range(100):

    # forward pass
    ypred = [mlp(x) for x in xs]
    loss = msl( ys, ypred )

    # backward pass
    for p in mlp.parameters():
        p.grad = 0.0
    loss.backward()

    # update
    update(mlp.parameters(), 0.09)

    print(k, loss.data)

0 0.0002443567551007867
1 0.00024414874299616223
2 0.00024394107640469232
3 0.00024373375447462055
4 0.00024352677635695608
5 0.0002433201412054968
6 0.00024311384817679682
7 0.00024290789643014533
8 0.00024270228512758644
9 0.000242497013433878
10 0.00024229208051649445
11 0.00024208748554563358
12 0.0002418832276941548
13 0.00024167930613762628
14 0.00024147572005427356
15 0.00024127246862498181
16 0.00024106955103330246
17 0.00024086696646540977
18 0.00024066471411011108
19 0.0002404627931588325
20 0.0002402612028056117
21 0.00024005994224708533
22 0.00023985901068246883
23 0.00023965840731355318
24 0.00023945813134470384
25 0.00023925818198283446
26 0.00023905855843741662
27 0.00023885925992043986
28 0.0002386602856464355
29 0.0002384616348324411
30 0.00023826330669800212
31 0.00023806530046514943
32 0.0002378676153584231
33 0.00023767025060480833
34 0.00023747320543379043
35 0.00023727647907727386
36 0.00023708007076963798
37 0.00023688397974768344
38 0.00023668820525064196
39 0.0

In [66]:
ypred

[Value(data=0.9935314269300363),
 Value(data=-0.9959978972806539),
 Value(data=-0.9903961999572064),
 Value(data=0.9913268502926068)]